In [ ]:
import os
import numpy as np
import mne
from glob import glob

In [ ]:
bids_root = r"C:\Users\Laura\OneDrive - Northwestern University\SoundBrain Lab - EAM1\data-bids"
print('BIDS root directory:', bids_root)
events_dir = r"C:\Users\Laura\Documents\PhD\Soundbrain lab\EAM\stimtrack-compare\python_v2"

# derivatives from status
deriv_dir_motor = os.path.join(bids_root, 'derivatives', 'cortical_filtered_motor')
if not os.path.exists(deriv_dir_motor):
    os.makedirs(deriv_dir_motor)

deriv_dir_status = os.path.join(bids_root, 'derivatives', 'cortical_filtered_status')
if not os.path.exists(deriv_dir_status):
    os.makedirs(deriv_dir_status) 

# deritivatives from stimtrack
deriv_dir = os.path.join(bids_root, 'derivatives', 'cortical_filtered')
if not os.path.exists(deriv_dir):
    os.makedirs(deriv_dir)

In [ ]:
task_list = ['active', 'passive', 'motor']
bl_start = 0.100
l_freq = 1
h_freq = 30
passive_hex = '#4477AA'
active_hex = '#CC6677'

In [ ]:
# create a list of epoches for each participant and each condition
act_eps = sorted(glob(deriv_dir+f'/sub-*_task-active_baseline-{bl_start}_filt-{l_freq}-{h_freq}_run-all_event-stimtrack_epo.fif'))
pas_eps = sorted(glob(deriv_dir+f'/sub-*_task-passive_baseline-{bl_start}_filt-{l_freq}-{h_freq}_run-all_event-stimtrack_epo.fif'))

results = []

for act_path, pas_path in zip(act_eps, pas_eps):
    sub_id = act_path.split('sub-')[1][:2]

    if sub_id == '08' or sub_id == '10':
        continue

    act_epochs = mne.read_epochs(act_path, verbose=False)
    pas_epochs = mne.read_epochs(pas_path, verbose=False)

    sfreq = act_epochs.info['sfreq']
    n_times = act_epochs.get_data().shape[2]

    prestim_sampples = int(round(abs(act_epochs.tmin) * sfreq))
    poststim_samples = n_times - prestim_sampples
    segment_points = [-prestim_sampples, poststim_samples]

    act_data = act_epochs.get_data()[:, 0, :].ravel()
    pas_data = pas_epochs.get_data()[:, 0, :].ravel()
    combined = np.concatenate((act_data, pas_data))

    print(f"sfreq: {sfreq}")
    print(f"tmin: {act_epochs.tmin}, tmax: {act_epochs.tmax}")
    print(f"prestim_samples: {prestim_sampples}, poststim_samples: {poststim_samples}")
    print(f"segment_size: {prestim_sampples + poststim_samples}")
    print(f"actual n_times: {act_epochs.get_data().shape[2]}")
    print(f"combined size: {combined.size}")
    print(f"combined % segment_size: {combined.size % (prestim_sampples + poststim_samples)}")

    snr_lb, snr_ub, snr_mean = erp_snr_ci(
        segment_dat=combined,
        segment_points=segment_points,
        percent_ci=0.90,
        num_bootstraps=9999,
        window=None,
        num_trials=0,
        num_conditions=2
    )

    results.append({
        'sub': sub_id,
        'snr_lb': snr_lb,
        'snr_ub': snr_ub,
        'snr_mean': snr_mean
    })

    print(f"sub-{sub_id} SNR: {snr_mean:.2f} (90% CI: [{snr_lb:.2f}, {snr_ub:.2f}])")



In [ ]:
def erp_snr_ci(
    segment_dat,
    segment_points,
    percent_ci=None,
    num_bootstraps=None,
    window=None,
    num_trials=0,
    num_conditions=0,
):
    """
    Computes signal-to-noise confidence intervals for individual subject ERP
    datasets using a bootstrap resampling procedure.

    Parameters
    ----------
    segment_dat : str or array-like
        Either a file path (str) to a text file containing a single column of
        concatenated EEG segments, or a 1-D array/list of the same data.
    segment_points : array-like of two ints
        [pre_stimulus, post_stimulus] sample counts, e.g. [-200, 800].
        The pre-stimulus value should be negative (or will be abs()'d).
    percent_ci : float or None
        Desired confidence interval as a value in (0, 1). Defaults to 0.90.
    num_bootstraps : int or None
        Number of bootstrap resamples. Defaults to 9999.
    window : array-like of two ints or None
        Post-stimulus time window (in samples) used for the SNR numerator,
        e.g. [0, 500]. Defaults to the full post-stimulus period.
    num_trials : int
        Number of randomly drawn segments per bootstrap ERP (S). Use this OR
        num_conditions, not both.
    num_conditions : int
        Number of experimental conditions. If provided (and num_trials == 0),
        S is set to round(total_segments / num_conditions).

    Returns
    -------
    snr_lb : float
        Lower bound of the SNR confidence interval (dB).
    snr_ub : float
        Upper bound of the SNR confidence interval (dB).
    snr_mean : float
        Mean SNR across all bootstrap resamples (dB).
    """

    prestim_points = abs(int(segment_points[0]))
    poststim_points = int(segment_points[1])
    segment_size = prestim_points + poststim_points

    # --- Confidence interval alpha ---
    if percent_ci is None:
        ci_alpha = 0.05
        percent_ci = 0.90
    else:
        ci_alpha = (1.0 - percent_ci) / 2.0

    # --- Number of bootstraps ---
    if num_bootstraps is None:
        num_bootstraps = 9999

    # --- SNR window (0-based indices, inclusive) ---
    if window is None:
        snr_start = prestim_points          # first post-stimulus sample
        snr_end = prestim_points + poststim_points  # exclusive upper bound
    else:
        snr_start = prestim_points + int(window[0])
        snr_end = prestim_points + int(window[1]) + 1  # make inclusive like MATLAB

    # --- Load data ---
    print()
    if isinstance(segment_dat, str):
        print(segment_dat)
        print("     Loading...")
        data = np.loadtxt(segment_dat)
    else:
        data = np.asarray(segment_dat, dtype=float).ravel()

    # --- Find number of segments ---
    total_points = data.size
    if total_points % segment_size != 0:
        raise ValueError(
            "ERROR: Number of segments and segment size do not add up. Exiting..."
        )
    num_segs = total_points // segment_size
    print(f"     {num_segs} segments found (N={num_segs}).")

    # --- Parse segments into 2-D array (num_segs × segment_size) ---
    eeg_segments = data.reshape(num_segs, segment_size)

    # --- Determine S (segments per bootstrap ERP) ---
    if num_trials > 0:
        s_value = int(num_trials)
    elif num_conditions > 0:
        s_value = int(round(num_segs / num_conditions))
    else:
        raise ValueError(
            "ERROR: Number of trials for bootstrap ERP calculations not specified.\n"
            "    A desired number of trials OR the number of conditions "
            "(for mean number of trials) must be given."
        )
    print(f"     {s_value} segments per bootstrap ERP (S={s_value}).")

    # --- Bootstrap resampling ---
    print(f"     Computing {percent_ci * 100:.1f}% CI from {num_bootstraps} bootstraps...")

    rng = np.random.default_rng()  # reproducible: pass a seed if desired
    rect_boot_erps = np.zeros((num_bootstraps, segment_size))

    for i in range(num_bootstraps):
        # Randomly select S segments with replacement
        indices = rng.integers(0, num_segs, size=s_value)
        sample = eeg_segments[indices, :]          # shape: (S, segment_size)

        # Mean waveform across selected segments
        erp = sample.mean(axis=0)

        # Baseline correction (subtract mean of pre-stimulus period)
        erp = erp - erp[:prestim_points].mean()

        # Rectify (absolute value via sqrt of squares, matching MATLAB)
        erp = np.sqrt(erp ** 2)

        rect_boot_erps[i, :] = erp

    # --- Compute SNR for each bootstrap ERP ---
    post_mean = rect_boot_erps[:, snr_start:snr_end].mean(axis=1)
    pre_mean  = rect_boot_erps[:, :prestim_points].mean(axis=1)
    snr_dist  = 20.0 * np.log10(post_mean / pre_mean)

    # Sort the distribution
    snr_dist = np.sort(snr_dist)

    # --- Confidence interval bounds ---
    lb_idx = int(round(ci_alpha * snr_dist.size)) - 1          # convert to 0-based
    ub_idx = int(round((1.0 - ci_alpha) * snr_dist.size)) - 1

    # Clamp to valid range
    lb_idx = max(0, min(lb_idx, snr_dist.size - 1))
    ub_idx = max(0, min(ub_idx, snr_dist.size - 1))

    snr_lb   = snr_dist[lb_idx]
    snr_ub   = snr_dist[ub_idx]
    snr_mean = snr_dist.mean()

    print()
    print(f"     SNR Lower Bound = {snr_lb:.4f} dB")
    print()
    print(f"     SNR Upper Bound = {snr_ub:.4f} dB")
    print()
    print(f"     SNR Mean = {snr_mean:.4f} dB")
    print()
    print()

    return snr_lb, snr_ub, snr_mean